In [0]:
df = spark.table("raw.dbt_billingshield.fct_provider_risk")
print(f"Rows: {df.count()}")
print(f"Columns: {len(df.columns)}")
df.show(3, truncate = False)

Rows: 236681
Columns: 20
+-----------+------------------+-----+-------------+--------+-----------+----------------+-------------------+--------------+------------+----------+----------------+---------------------+----------+----------+-----------------+-------------+--------------+---------+--------------------------+
|provider_id|specialty         |state|city         |zip_code|entity_code|total_procedures|total_beneficiaries|total_services|total_billed|total_paid|avg_charge_ratio|avg_billing_intensity|max_zscore|avg_zscore|unique_procedures|facility_rate|specialty_rank|risk_tier|dbt_processed_at          |
+-----------+------------------+-----+-------------+--------+-----------+----------------+-------------------+--------------+------------+----------+----------------+---------------------+----------+----------+-----------------+-------------+--------------+---------+--------------------------+
|1073992145 |Addiction Medicine|NY   |Staten Island|10305   |I          |3               |

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_features = spark.table("raw.dbt_billingshield.fct_provider_risk")

# Construct fraud label from statistical thresholds
df_ml = df_features.withColumn(
    "fraud_flag",
    F.when(F.col("max_zscore") >= 3, 1).otherwise(0)
)

# Select ML feature columns
feature_cols = [
    "provider_id",
    "avg_charge_ratio",
    "avg_billing_intensity", 
    "max_zscore",
    "avg_zscore",
    "unique_procedures",
    "facility_rate",
    "total_procedures",
    "total_services",
    "total_beneficiaries",
    "fraud_flag"
]

df_ml = df_ml.select(feature_cols)

print(f"Total providers: {df_ml.count():,}")
print(f"Fraud (1): {df_ml.filter(F.col('fraud_flag')==1).count():,}")
print(f"Non-fraud (0): {df_ml.filter(F.col('fraud_flag')==0).count():,}")

Total providers: 236,681
Fraud (1): 17,891
Non-fraud (0): 218,790


In [0]:
%pip install xgboost

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
import xgboost as xgb
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

# Convert to Pandas
pdf = df_ml.toPandas()

# Fill nulls
pdf = pdf.fillna(0)

# Features and label
feature_cols = [
    "avg_charge_ratio", "avg_billing_intensity", "max_zscore",
    "avg_zscore", "unique_procedures", "facility_rate",
    "total_procedures", "total_services", "total_beneficiaries"
]

X = pdf[feature_cols]
y = pdf["fraud_flag"]
groups = pdf["provider_id"]

print(f"Feature matrix shape: {X.shape}")
print(f"Class balance: {y.value_counts().to_dict()}")
print(f"scale_pos_weight: {round((y==0).sum() / (y==1).sum(), 2)}")

Feature matrix shape: (236681, 9)
Class balance: {0: 218790, 1: 17891}
scale_pos_weight: 12.23


In [0]:
from sklearn.model_selection import GroupShuffleSplit
import xgboost as xgb
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

# NPI-level split — prevents data leakage
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")

# Train XGBoost
model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=12.23,
    eval_metric='aucpr',
    random_state=42,
    verbosity=0
)

model.fit(X_train, y_train)
print("Model trained.")

Train: 189,344 | Test: 47,337
Model trained.


In [0]:
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("=== Model Evaluation ===")
print(classification_report(y_test, y_pred, target_names=['Normal', 'Fraud']))
print(f"AUC-ROC:  {roc_auc_score(y_test, y_prob):.4f}")
print(f"AUC-PR:   {average_precision_score(y_test, y_prob):.4f}")

=== Model Evaluation ===
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00     43737
       Fraud       0.98      1.00      0.99      3600

    accuracy                           1.00     47337
   macro avg       0.99      1.00      0.99     47337
weighted avg       1.00      1.00      1.00     47337

AUC-ROC:  1.0000
AUC-PR:   0.9998


In [0]:
import joblib
import pandas as pd

# Feature importance
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("=== Feature Importance ===")
print(importance.to_string(index=False))

# Save model
joblib.dump(model, '/tmp/xgboost_fraud_v1.pkl')
print("\nModel saved to /tmp/xgboost_fraud_v1.pkl")

=== Feature Importance ===
              feature  importance
           max_zscore    0.996078
        facility_rate    0.000581
       total_services    0.000540
avg_billing_intensity    0.000522
     avg_charge_ratio    0.000509
           avg_zscore    0.000502
  total_beneficiaries    0.000459
    unique_procedures    0.000425
     total_procedures    0.000384

Model saved to /tmp/xgboost_fraud_v1.pkl


In [0]:
%pip install shap

import shap
import pandas as pd

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Global feature importance via SHAP
shap_importance = pd.DataFrame({
    'feature': feature_cols,
    'shap_importance': abs(shap_values).mean(axis=0)
}).sort_values('shap_importance', ascending=False)

print("=== SHAP Feature Importance ===")
print(shap_importance.to_string(index=False))

# Per-provider rationale string for top 3 test providers
print("\n=== Sample Provider Risk Rationale ===")
for i in range(3):
    top_features = sorted(
        zip(feature_cols, shap_values[i]),
        key=lambda x: abs(x[1]), reverse=True
    )[:3]
    rationale = ", ".join([f"{f} (impact: {v:.3f})" for f, v in top_features])
    print(f"Provider {i+1}: {rationale}")

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
=== SHAP Feature Importance ===
              feature  shap_importance
           max_zscore        11.370480
           avg_zscore         0.376998
     avg_charge_ratio         0.225292
  total_beneficiaries         0.187252
       total_services         0.174342
    unique_procedures         0.139698
        facility_rate         0.125559
avg_billing_intensity         0.096561
     total_procedures         0.036948

=== Sample Provider Risk Rationale ===
Provider 1: max_zscore (impact: -11.310), avg_zscore (impact: -0.450), total_services (impact: -0.326)
Provider 2: max_zscore (impact: -11.430), avg_zscore (impact: -0.626), avg_billing_intensity (impact: -0.160)
Provider 3: max_zscore (impact: -10.963), total_services (impact: -0.823), avg_charge_ratio (impact: -0.396)


In [0]:
# Add predictions back to Spark dataframe and save as Gold table
import pandas as pd
import numpy as np

pdf["risk_score"] = model.predict_proba(X)[:, 1]
pdf["predicted_fraud"] = model.predict(X)

df_results = spark.createDataFrame(pdf)

df_results.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("raw.gold.ml_provider_scores")

print(f"ML scores table written: {df_results.count():,} providers")

ML scores table written: 236,681 providers
